In [ ]:
# Enable autoreload to automatically reload modules before executing code
%load_ext autoreload
%autoreload 2

# Forecasting Strategies with Machine Learning Models

When using machine learning (ML) models for time series forecasting, it's important to choose a **forecasting strategy** that matches your goals and the nature of your data. These strategies determine how the model generates forecasts for multiple future time steps (the forecast horizon). Let's break down the most common approaches:

---

## 1. **Direct Forecasting**

**What is it?**  
In the **direct** strategy, you train a separate model for each forecast horizon. For example, to predict the value at $t+1$, $t+2$, ..., $t+h$, you build $h$ different models—each specialized for a specific step ahead.

**How it works:**  
- Model 1 predicts $y_{t+1}$ using data up to time $t$.
- Model 2 predicts $y_{t+2}$ using data up to time $t$.
- ...and so on, up to $y_{t+h}$.

**Pros:**  
- Each model can focus on the unique patterns relevant to its specific horizon.
- Often yields better accuracy for longer horizons.

**Cons:**  
- Requires training and maintaining multiple models.
- Can be computationally expensive.

---

## 2. **Recursive (Iterated) Forecasting**

**What is it?**  
The **recursive** (or **iterated**) strategy uses a single model to predict the next time step. To forecast further ahead, it feeds its own predictions back as inputs.

**How it works:**  
- Train one model to predict $y_{t+1}$ from past values.
- To get $y_{t+2}$, use the predicted $y_{t+1}$ as an input, and so on.

**Pros:**  
- Only one model to train and maintain.
- Simple to implement.

**Cons:**  
- Errors can accumulate as predictions are fed back (error propagation).
- May perform poorly for long horizons.

---

## 3. **DirRec (Direct-Recursive) Forecasting**

**What is it?**  
**DirRec** combines the direct and recursive approaches. For each forecast horizon, you train a separate model, but each model can use previous predictions as features.

**How it works:**  
- Model for $y_{t+1}$ uses only historical data.
- Model for $y_{t+2}$ uses historical data and the predicted $y_{t+1}$.
- Model for $y_{t+3}$ uses historical data and predictions for $y_{t+1}$ and $y_{t+2}$.
- ...and so on.

**Pros:**  
- Can capture dependencies between future steps.
- Balances flexibility and complexity.

**Cons:**  
- Still requires multiple models.
- Can still suffer from error propagation, but often less than pure recursive.

---

## 4. **Multi-Output (Multi-Horizon) Forecasting**

**What is it?**  
In the **multi-output** (or **multi-horizon**) strategy, a single model is trained to predict all future steps at once.

**How it works:**  
- The model outputs a vector: $[\hat{y}_{t+1}, \hat{y}_{t+2}, ..., \hat{y}_{t+h}]$.

**Pros:**  
- Captures dependencies between future time steps.
- Only one model to train.

**Cons:**  
- Can be more complex to set up.
- May require more data to train effectively.

---

## **Which Strategy Should You Use?**

- **Short horizons:** Recursive or direct strategies often suffice.
- **Long horizons:** Direct, DirRec, or multi-output strategies can reduce error accumulation.
- **Complex dependencies:** Multi-output or DirRec can better capture relationships between future steps.

---

## **Summary Table**

| Strategy      | # Models | Error Propagation | Captures Step Dependencies | Complexity |
|---------------|----------|-------------------|---------------------------|------------|
| Direct        | $h$      | No                | No                        | Moderate   |
| Recursive     | 1        | Yes               | No                        | Low        |
| DirRec        | $h$      | Some              | Yes                       | High       |
| Multi-Output  | 1        | No                | Yes                       | High       |

---

**In practice:**  
The best strategy depends on your data, forecast horizon, and computational resources. Many modern libraries (like `mlforecast` and `statsforecast`) support several of these strategies, making it easy to experiment and find what works best for your time series problem.

In [ ]:
import polars as pl
import plotly.express as px
import seaborn as sns
from utilsforecast.plotting import plot_series
from statsforecast import StatsForecast
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from utilsforecast.losses import *
from utilsforecast.evaluation import evaluate
from mlforecast import MLForecast
from mlforecast.target_transforms import (
    LocalStandardScaler,
    LocalMinMaxScaler,
    Differences,
)
from plotting_utils import (
    plotly_series as plot_series,
    plot_acf,
    plot_residuals_diagnostic,
)
from summary_utils import get_fitted_residuals
from statsmodels.stats.diagnostic import acorr_ljungbox
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

import plotly.graph_objects as go


In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV
from xgboost import XGBRegressor

In [ ]:
data = pl.read_parquet(
    "data/london_smart_meters/preprocessed/london_smart_meters_merged_block_0-7.parquet"
)
timestamp = data.group_by("LCLid").agg(
    pl.datetime_range(
        start=pl.col("start_timestamp"),
        end=pl.col("start_timestamp").dt.offset_by(
            pl.format("{}m", pl.col("series_length").sub(1).mul(30))
        ),
        interval="30m",
    ).alias("ds"),
)
data = timestamp.join(data, on="LCLid", how="inner").rename(
    {"LCLid": "unique_id", "energy_consumption": "y"}
)
data.head(5)

In [ ]:
id_ = "unique_id"
time_ = "ds"
target_ = "y"
id_col = pl.col(id_)
time_col = pl.col(time_)
target_col = pl.col(target_)

In [ ]:
data = (
    data.filter(pl.col("file").eq("block_7"))
    .select(
        [
            time_,
            id_,
            target_,
            # "Acorn",
            # "Acorn_grouped",
            # "holidays",
            # "visibility",
            # "windBearing",
            # "temperature",
            # "dewPoint",
            # "pressure",
            # "apparentTemperature",
            # "windSpeed",
            # "precipType",
            # "icon",
            # "humidity",
            # "summary",
        ]
    )
    .explode(
        [
            time_,
            target_,
            # "holidays",
            # "visibility",
            # "windBearing",
            # "temperature",
            # "dewPoint",
            # "pressure",
            # "apparentTemperature",
            # "windSpeed",
            # "precipType",
            # "icon",
            # "humidity",
            # "summary",
        ]
    )
)
data = data.select(time_, id_, target_)
data.head()

In [ ]:
selected_id = "MAC000193"
data = data.filter(id_col.eq(selected_id)).with_columns(
    target_col.forward_fill().backward_fill()
)
data.head()

# Deep Dive: The Direct Forecasting Strategy

## What is the Direct Strategy?

The **direct strategy** is a foundational approach for multi-step time series forecasting. Instead of using a single model to predict all future values, the direct strategy trains a **separate model for each forecast horizon**. This means if you want to forecast $h$ steps ahead, you will build $h$ different models—each one specialized for a specific time step in the future.

---

## How Does It Work?

Suppose you have a time series $y_t$ and you want to forecast the next $h$ values: $y_{t+1}, y_{t+2}, ..., y_{t+h}$.

- **Model 1** is trained to predict $y_{t+1}$ using data up to time $t$.
- **Model 2** is trained to predict $y_{t+2}$ using data up to time $t$.
- ...
- **Model $h$** is trained to predict $y_{t+h}$ using data up to time $t$.

Each model is independent and focuses on learning the patterns relevant to its specific forecast horizon.

**Mathematically:**

$$
\begin{align*}
\hat{y}_{t+1} &= f_1(\text{features at } t) \\
\hat{y}_{t+2} &= f_2(\text{features at } t) \\
&\vdots \\
\hat{y}_{t+h} &= f_h(\text{features at } t)
\end{align*}
$$

---

## Why Use the Direct Strategy?

- **Specialization:** Each model can focus on the unique challenges of its forecast horizon. For example, predicting tomorrow's temperature may require different information than predicting a week ahead.
- **Reduced Error Propagation:** Since each model is independent, errors made at earlier steps do **not** affect later predictions.
- **Flexibility:** You can use different features or even different algorithms for each horizon if needed.

---

## Real-World Analogy

Imagine you’re planning meals for the next week. Instead of making a single plan and adjusting it each day (which could lead to compounding mistakes), you create a separate, detailed plan for each day. This way, if you make a mistake planning Monday’s meal, it doesn’t affect Tuesday’s or Wednesday’s plans.

---

## Step-by-Step Example

Let’s say you want to forecast **energy consumption** for the next 48 half-hour periods (1 day ahead) using the direct strategy:

1. **Prepare your features:** For each time $t$, create features from historical data (lags, rolling means, date features, etc.).
2. **Train 48 models:** Each model predicts the value at a specific future time (e.g., 30 minutes ahead, 1 hour ahead, ..., 24 hours ahead).
3. **Make predictions:** At prediction time, use the latest available data to generate all 48 forecasts at once.

---

## Pros and Cons

| Pros                                         | Cons                                  |
|-----------------------------------------------|---------------------------------------|
| No error propagation between steps            | Requires training $h$ models          |
| Each model can be highly specialized          | More computationally intensive        |
| Often better for long-horizon forecasting     | Harder to maintain many models        |

---

## Common Beginner Questions

**Q: Why not just use one model for all steps?**  
A: Using one model (recursive strategy) is simpler, but errors can accumulate as predictions are fed back into the model. The direct strategy avoids this by making each prediction independently.

**Q: Do I need to create different features for each model?**  
A: You can use the same features for all models, but sometimes it helps to customize features for each horizon.

**Q: Is the direct strategy always better?**  
A: Not always. It works well for longer horizons but can be overkill for short-term forecasts or when computational resources are limited.

---

## Summary

The **direct strategy** is a robust and interpretable approach for multi-step forecasting, especially when you care about accuracy at specific future points and want to avoid error accumulation. It’s widely supported in modern libraries like `mlforecast`, making it accessible even for beginners.

---

**Next:** We'll see how to implement the direct strategy in practice using the `mlforecast` library and our energy consumption dataset.

In [ ]:
from mlforecast.lag_transforms import (
    RollingMean,
    RollingStd,
    SeasonalRollingMean,
    SeasonalRollingStd,
    ExponentiallyWeightedMean,
)

lags = [1, 2, 48, 336]
lag_transforms = {
    1: [
        RollingMean(window_size=3),
        RollingMean(window_size=6),
        RollingMean(window_size=12),
        RollingMean(window_size=48),
        RollingStd(window_size=3),
        RollingStd(window_size=6),
        RollingStd(window_size=12),
        RollingStd(window_size=48),
        ExponentiallyWeightedMean(alpha=0.25),
    ],
    48: [
        RollingMean(window_size=7),
        RollingMean(window_size=14),
        RollingStd(window_size=7),
        RollingStd(window_size=14),
        SeasonalRollingMean(season_length=48, window_size=3),
        SeasonalRollingStd(season_length=48, window_size=3),
    ],
    336: [
        RollingMean(window_size=4),
        RollingMean(window_size=8),
        RollingStd(window_size=4),
        RollingStd(window_size=8),
        SeasonalRollingMean(season_length=336, window_size=3),
        SeasonalRollingStd(season_length=336, window_size=3),
    ],
}

In [ ]:
date_features = [
    "month",
    "quarter",
    "week",
    "day",
    "weekday",
    "hour",
    "minute",
]

In [ ]:
mlf = MLForecast(
    models=[
        RidgeCV(),
        XGBRegressor(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            tree_method="hist",
        ),
    ],
    freq="30m",
    lags=lags,
    lag_transforms=lag_transforms,
    date_features=date_features,
)

# We'll use cross-validation to evaluate the model's forecasting performance.
y_hat = mlf.cross_validation(
    data,
    h=48,  # Forecast the next 48 half-hour periods (1 day ahead)
    step_size=1,
    n_windows=1,
    fitted=True,
    max_horizon=48,
).drop("cutoff")

In [ ]:
plot_series(data, y_hat, max_insample_length=200)

In [ ]:
from functools import partial

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

# Deep Dive: The Recursive (Iterated) Forecasting Strategy

## What is the Recursive Strategy?

The **recursive strategy**—also known as the **iterated** or **one-step-ahead** approach—is a classic method for multi-step time series forecasting. Instead of training a separate model for each forecast horizon, you train **one model** to predict the next time step. To forecast further into the future, you feed the model’s own predictions back as inputs.

---

## How Does It Work?

Suppose you have a time series $y_t$ and want to forecast the next $h$ values: $y_{t+1}, y_{t+2}, ..., y_{t+h}$.

- **Step 1:** Train a model to predict $y_{t+1}$ using historical data up to time $t$.
- **Step 2:** Use the model to predict $y_{t+1}$.
- **Step 3:** Use the predicted $y_{t+1}$ (instead of the true value) as an input to predict $y_{t+2}$.
- **Repeat:** Continue this process, each time using the most recent prediction as input for the next forecast step.

**Mathematically:**

$$
\begin{align*}
\hat{y}_{t+1} &= f(\text{features at } t) \\
\hat{y}_{t+2} &= f(\text{features at } t+1, \hat{y}_{t+1}) \\
\hat{y}_{t+3} &= f(\text{features at } t+2, \hat{y}_{t+2}) \\
&\vdots \\
\hat{y}_{t+h} &= f(\text{features at } t+h-1, \hat{y}_{t+h-1})
\end{align*}
$$

---

## Why Use the Recursive Strategy?

- **Simplicity:** Only one model needs to be trained and maintained.
- **Efficiency:** Less computationally intensive than training multiple models.
- **Consistency:** The same model logic is applied at each forecast step.

---

## Real-World Analogy

Imagine you’re writing a story, one sentence at a time. After each sentence, you read what you just wrote and use it to decide what comes next. If you make a mistake in one sentence, it might affect the following sentences—this is similar to how errors can accumulate in the recursive strategy.

---

## Step-by-Step Example

Let’s say you want to forecast **energy consumption** for the next 48 half-hour periods (1 day ahead):

1. **Prepare your features:** For each time $t$, create features from historical data (lags, rolling means, date features, etc.).
2. **Train one model:** The model learns to predict the next value ($y_{t+1}$) from the current features.
3. **Make predictions:**  
    - Predict $y_{t+1}$ using the latest available data.
    - Use $\hat{y}_{t+1}$ as input to predict $\hat{y}_{t+2}$, and so on, until you reach $y_{t+48}$.

---

## Pros and Cons

| Pros                                   | Cons                                      |
|-----------------------------------------|-------------------------------------------|
| Only one model to train and maintain    | Errors can accumulate (error propagation) |
| Simple to implement                    | May perform poorly for long horizons      |
| Computationally efficient               | Each step depends on previous predictions |

---

## Common Beginner Questions

**Q: Why do errors accumulate in the recursive strategy?**  
A: Because each prediction is used as input for the next, any mistake made early on can affect all subsequent forecasts. This is known as **error propagation**.

**Q: When is the recursive strategy a good choice?**  
A: It works well for short forecast horizons or when you need a quick, simple solution.

**Q: Can I use the same features for each step?**  
A: Yes! The model always uses the same set of features, but after the first step, some features (like lagged values) will be based on predictions rather than actual data.

---

## Summary

The **recursive strategy** is a straightforward and efficient approach for multi-step forecasting, especially when you want to keep things simple and the forecast horizon is short. However, be mindful of error propagation, which can degrade accuracy for longer horizons. Many libraries, including `mlforecast`, make it easy to implement this strategy.

---

**Next:** We’ll explore how to implement the recursive strategy in practice and compare its performance to the direct approach.

In [ ]:
mlf = MLForecast(
    models=[
        RidgeCV(),
        XGBRegressor(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            tree_method="hist",
        ),
    ],
    freq="30m",
    lags=lags,
    lag_transforms=lag_transforms,
    date_features=date_features,
)

# We'll use cross-validation to evaluate the model's forecasting performance.
y_hat = mlf.cross_validation(
    data,
    h=48,  # Forecast the next 48 half-hour periods (1 day ahead)
    step_size=1,
    n_windows=1,
    fitted=True,
).drop("cutoff")

In [ ]:
plot_series(data, y_hat, max_insample_length=200)

In [ ]:
from functools import partial

metrics = [
    mae,
    mse,
    rmse,
    mape,
    smape,
    partial(mase, seasonality=48),
]
evaluate(
    y_hat,
    metrics=metrics,
    train_df=data.select([id_, time_, target_]),
)

# Deep Dive: The DirRec (Direct-Recursive) Forecasting Strategy

## What is the DirRec Strategy?

The **DirRec strategy** (short for **Direct-Recursive**) is a hybrid approach for multi-step time series forecasting. It combines the strengths of both the direct and recursive strategies:

- Like the direct strategy, it trains a separate model for each forecast horizon.
- Like the recursive strategy, each model can use previous predictions as features for future steps.

This allows each model to specialize for its forecast step while also capturing dependencies between future values.

---

## How Does DirRec Work?

Suppose you want to forecast $h$ steps ahead for a time series $y_t$:

- **Model 1** predicts $y_{t+1}$ using only historical data.
- **Model 2** predicts $y_{t+2}$ using historical data **and** the predicted $y_{t+1}$.
- **Model 3** predicts $y_{t+3}$ using historical data and predictions for $y_{t+1}$ and $y_{t+2}$.
- ...and so on, up to $y_{t+h}$.

**Mathematically:**

$$
\begin{align*}
\hat{y}_{t+1} &= f_1(\text{features at } t) \\
\hat{y}_{t+2} &= f_2(\text{features at } t, \hat{y}_{t+1}) \\
\hat{y}_{t+3} &= f_3(\text{features at } t, \hat{y}_{t+1}, \hat{y}_{t+2}) \\
&\vdots \\
\hat{y}_{t+h} &= f_h(\text{features at } t, \hat{y}_{t+1}, ..., \hat{y}_{t+h-1})
\end{align*}
$$

---

## Why Use the DirRec Strategy?

- **Captures Step Dependencies:** By using previous predictions as features, DirRec can model relationships between future time steps.
- **Balances Complexity:** It avoids the full error propagation of recursive strategies and the total independence of direct strategies.
- **Flexible:** Each model can use different features or algorithms, tailored for its horizon.

---

## Real-World Analogy

Imagine planning a multi-day trip. For each day, you make a plan, but you also consider what happened on previous days (e.g., if you got delayed or changed your route). Each day's plan is unique but informed by earlier outcomes.

---

## Step-by-Step Example

Let’s forecast **energy consumption** for the next 48 half-hour periods (1 day ahead) using DirRec:

1. **Prepare features:** For each time $t$, create features from historical data.
2. **Train 48 models:**  
    - Model 1 uses only historical data.
    - Model 2 uses historical data and Model 1’s prediction.
    - Model 3 uses historical data and predictions from Models 1 and 2.
    - ...and so on.
3. **Make predictions:**  
    - Predict $y_{t+1}$ with Model 1.
    - Use $\hat{y}_{t+1}$ as a feature for Model 2 to predict $y_{t+2}$.
    - Continue this process for all 48 steps.

---

## Pros and Cons

| Pros                                         | Cons                                  |
|-----------------------------------------------|---------------------------------------|
| Captures dependencies between forecast steps  | More complex to implement             |
| Reduces error propagation vs. recursive       | Requires training $h$ models          |
| Flexible feature engineering                  | Still some error propagation possible |

---

## Common Beginner Questions

**Q: How is DirRec different from direct or recursive?**  
A: DirRec uses multiple models (like direct), but each model can use previous predictions as features (like recursive).

**Q: Does DirRec always outperform other strategies?**  
A: Not always. Its strength is in capturing dependencies between steps, but it can still suffer from error propagation and is more complex to set up.

**Q: Can I use the same features for all models?**  
A: You can, but DirRec allows you to add previous predictions as features for each step.

---

## Summary

The **DirRec strategy** is a powerful and flexible approach for multi-step forecasting, especially when future values are likely to depend on each other. It’s a great choice when you want to balance specialization and dependency modeling.

---

In [ ]:
from sktime.forecasting.compose import make_reduction
from sktime.forecasting.model_selection import temporal_train_test_split
from sktime.performance_metrics.forecasting import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)
from sktime.forecasting.base import ForecastingHorizon
from sklearn.linear_model import RidgeCV
import pandas as pd

# DirRec (Direct-Recursive) strategy is not natively implemented in mlforecast,
# but sktime provides a convenient interface for this approach.


# Convert polars DataFrame to pandas Series for sktime compatibility
y_pd = data.to_pandas().set_index(time_)[target_]

# Split data into train and test sets
y_train, y_test = temporal_train_test_split(y_pd, test_size=48)
fh = ForecastingHorizon(y_test.index, is_relative=False)

# Create a DirRec forecaster using RidgeCV as the base regressor
dirrec_forecaster = make_reduction(
    RidgeCV(),
    strategy="dirrec",
    window_length=336,  # Use 7 lags (1 week) as features
)

# Fit the DirRec forecaster
dirrec_forecaster.fit(y_train, fh=fh)

# Forecast the next 48 steps
y_pred = dirrec_forecaster.predict(fh)

# Evaluate forecast accuracy
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

print(f"DirRec RidgeCV MAE: {mae:.4f}")
print(f"DirRec RidgeCV MSE: {mse:.4f}")
print(f"DirRec RidgeCV MAPE: {mape:.4f}")

# Plot actual vs predicted using plotly for interactivity
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Scatter(x=y_test.index, y=y_test.values, mode="lines+markers", name="Actual")
)
fig.add_trace(
    go.Scatter(
        x=y_pred.index,
        y=y_pred.values,
        mode="lines+markers",
        name="DirRec RidgeCV Forecast",
    )
)
fig.update_layout(
    title="DirRec Forecast vs Actual (RidgeCV)",
    xaxis_title="Time",
    yaxis_title="Energy Consumption",
    legend_title="Legend",
    template="plotly_white",
)
fig.show()

# Deep Dive: The Multi-Output (Multi-Horizon) Forecasting Strategy

## What is the Multi-Output Strategy?

The **multi-output strategy** (also called **multi-horizon** forecasting) is a powerful approach for predicting multiple future time steps in a time series **all at once**. Instead of training separate models for each forecast horizon (like the direct strategy) or looping predictions step-by-step (like the recursive strategy), the multi-output approach uses a **single model** that outputs a vector of future values.

---

## How Does It Work?

Suppose you have a time series $y_t$ and want to forecast the next $h$ values: $y_{t+1}, y_{t+2}, ..., y_{t+h}$.

- You train **one model** that takes the current and historical data as input and produces a vector of $h$ forecasts as output:
    $$
    [\hat{y}_{t+1}, \hat{y}_{t+2}, ..., \hat{y}_{t+h}] = f(\text{features at } t)
    $$

- This model learns to predict all future steps **simultaneously**, capturing dependencies and patterns across the entire forecast horizon.

---

## Why Use the Multi-Output Strategy?

- **Captures Step Dependencies:** The model can learn relationships between future time steps (e.g., if a spike at $t+1$ usually leads to a dip at $t+2$).
- **Efficiency:** Only one model to train and maintain, regardless of the forecast horizon.
- **Consistency:** All forecasts are generated together, ensuring they are mutually coherent.

---

## Real-World Analogy

Imagine you’re planning a week’s worth of meals. Instead of planning each day separately or adjusting each day based on the previous one, you sit down and plan the entire week in one go, making sure the meals make sense together (e.g., leftovers from Monday can be used on Tuesday).

---

## Step-by-Step Example

Let’s say you want to forecast **energy consumption** for the next 48 half-hour periods (1 day ahead):

1. **Prepare your features:** For each time $t$, create features from historical data (lags, rolling means, date features, etc.).
2. **Train one model:** The model learns to output a vector of 48 values, each corresponding to a specific future time step.
3. **Make predictions:** At prediction time, use the latest available data to generate all 48 forecasts at once.

---

## Pros and Cons

| Pros                                         | Cons                                      |
|-----------------------------------------------|-------------------------------------------|
| Captures dependencies between forecast steps  | Can require more data to train effectively|
| Only one model to train and maintain          | Model architecture can be more complex    |
| Consistent, coherent forecasts                | Not all ML libraries support this natively|

---

## Common Beginner Questions

**Q: How is this different from the direct strategy?**  
A: The direct strategy trains a separate model for each step ahead. The multi-output strategy uses a single model to predict all steps at once, allowing it to learn relationships between future values.

**Q: What types of models can be used?**  
A: Many neural network architectures (like RNNs, LSTMs, and Transformers) and some tree-based models (like multi-output regressors) can be used for this strategy.

**Q: Is it always better?**  
A: Not always. It works best when future values are strongly related, but it may require more data and careful tuning.

---

## Mathematical Formulation

The multi-output model learns a function:
$$
[\hat{y}_{t+1}, \hat{y}_{t+2}, ..., \hat{y}_{t+h}] = f(\mathbf{x}_t)
$$
where $\mathbf{x}_t$ represents the features at time $t$ (such as past values, date features, etc.).

---

## Summary

The **multi-output strategy** is a modern, flexible approach for multi-step forecasting. By predicting all future values at once, it can capture complex dependencies and produce consistent forecasts. This strategy is especially popular in deep learning and is supported by libraries like `neuralforecast` from Nixtla.

---

> **Note:** We will leave the practical implementation of the multi-output strategy for the future, once we start working with neural networks and the `neuralforecast` library. This will allow us to fully leverage the strengths of this approach in a hands-on, beginner-friendly way.